In [1]:
import numpy as np

The convertion from $K, \Lambda$ in a sliding mode controller to a PD controller:

$$	K_d = K + m\Lambda, \qquad K_p = K \Lambda $$

In [2]:
def smc_to_pid(K, Lambda, m=2.0):
    """Convert Sliding Mode Control (SMC) gains to Proportional-Integral-Derivative (PID) gains. Assuming K, Lambda are 3 by 1 numpy arrays."""
    K_p = K * Lambda
    K_d = K + m * Lambda
    return K_p, K_d

In [33]:
K_p, K_d = smc_to_pid(np.array([3.5, 3.5, 3.5]), np.array([0.8, 0.8, 0.8]))
print(K_p, K_d)

[2.8 2.8 2.8] [5.1 5.1 5.1]


In [17]:
def pid_to_smc(K_p, K_d, m=2.0, branch='larger'):
    """Recover Sliding Mode Control gains (K, Lambda) from PID gains.
    """
    K_p = np.asarray(K_p, dtype=float).reshape(3)
    K_d = np.asarray(K_d, dtype=float).reshape(3)
    if np.any(K_p <= 0) or np.any(K_d <= 0):
        raise ValueError('K_p and K_d must be strictly positive for SMC recovery.')
    discriminant = K_d ** 2 - 4.0 * m * K_p
    if np.any(discriminant < 0):
        raise ValueError(f'No positive SMC gains satisfy these PID gains ({discriminant} < 0).')
    sqrt_disc = np.sqrt(discriminant)
    if branch == 'larger':
        K = 0.5 * (K_d + sqrt_disc)
    elif branch == 'smaller':
        K = 0.5 * (K_d - sqrt_disc)
    else:
        raise ValueError("branch must be either 'larger' or 'smaller'")
    Lambda = (K_d - K) / m
    if np.any(Lambda <= 0) or np.any(K <= 0):
        raise ValueError('Recovered gains must remain positive. Try a different branch.')
    return K, Lambda


In [22]:
K_p = 1 * np.array([1.0, 1.0, 1.0])
K_d = 5 * np.array([1.0, 1.0, 1.0])
K_rec, Lambda_rec = pid_to_smc(K_p, K_d, m=2.0, branch='larger')
print('Recovered K:', K_rec)
print('Recovered Lambda:', Lambda_rec)
K_p_check, K_d_check = smc_to_pid(K_rec, Lambda_rec, m=2.0)
print('Check K_p:', K_p_check)
print('Check K_d:', K_d_check)


Recovered K: [4.56155281 4.56155281 4.56155281]
Recovered Lambda: [0.21922359 0.21922359 0.21922359]
Check K_p: [1. 1. 1.]
Check K_d: [5. 5. 5.]
